# Hướng dẫn Chạy mô hình Multimodal (Late Fusion) trên Google Colab
Notebook này đã được cấu hình tự động 100% để bạn có thể chạy từ đầu đến cuối mà không gặp lỗi.

**LƯU Ý TRƯỚC KHI CHẠY:**
Bạn cần đảm bảo đã tạo thư mục Lối tắt (Shortcut) tên là `SE365` trên Google Drive của bạn (trỏ từ link chia sẻ dữ liệu của nhóm). Nếu bạn đặt tên lối tắt khác, hãy sửa tên thư mục ở **Bước 3** và **Bước 3.5** nhé.


### BƯỚC 1: Mount Google Drive
Lệnh này sẽ yêu cầu bạn cấp quyền truy cập Google Drive. Chúng ta cần làm điều này để đọc dữ liệu gốc (5000 ảnh và CSV) thông qua Lối tắt (Shortcut) mà không cần tải lại file zip.


In [ ]:
# BƯỚC 1: Mount Google Drive
# (Để đọc trực tiếp thư mục chứa text/ và image/ mà không cần zip)
from google.colab import drive
drive.mount('/content/drive')

### BƯỚC 2: Clone mã nguồn và cài đặt thư viện
Tải phiên bản code mới nhất từ Github và cài đặt các thư viện cần thiết (PyTorch, Transformers, Timm, ...).


In [ ]:
# BƯỚC 2: Clone mã nguồn và cài đặt thư viện
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt

### BƯỚC 3: Nạp dữ liệu vào máy ảo Colab (QUAN TRỌNG)
Để GPU không phải chờ ổ cứng đọc qua mạng, chúng ta sẽ copy toàn bộ thư mục `data` từ Drive sang thẳng ổ cứng cục bộ của máy ảo Colab. Việc này sẽ mất khoảng 1-2 phút nhưng tăng tốc huấn luyện lên gấp nhiều lần.

> ✏️ **Chỗ cần sửa:** Nếu lối tắt của bạn không lưu ở thư mục gốc Drive hoặc có tên khác `SE365`, hãy sửa đường dẫn `/content/drive/MyDrive/SE365/data` bên dưới cho khớp.


In [ ]:
# BƯỚC 3: Sao chép dữ liệu từ Google Drive vào máy ảo Colab (Tăng tốc tối đa)
!rm -rf ./data
!cp -r /content/drive/MyDrive/SE365/data ./data

# Kiểm tra xem dữ liệu đã được copy chưa
!ls -la ./data

### BƯỚC 3.5: Cấu hình nơi lưu Checkpoint (Trọng số mô hình)
Để tránh bị mất kết quả khi Colab tự ngắt, đoạn code sau sẽ tạo một thư mục con trong `checkpoints/` trên Drive của bạn, với tên là thời gian hiện tại (Ví dụ: `20260611_083908`). 
Tất cả các mô hình Text, Image, Fusion sau khi train xong sẽ được tự động copy vào chung thư mục này.

> ✏️ **Chỗ cần sửa:** Nếu bạn muốn lưu vào thư mục khác, hãy sửa biến `drive_ckpt_path`.


In [ ]:
# BƯỚC 3.5: Khởi tạo thư mục lưu trữ cho toàn bộ phiên chạy này
# Đảm bảo tất cả mô hình train trong hôm nay đều nằm chung một thư mục
import os
import datetime

run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
drive_ckpt_path = f'/content/drive/MyDrive/SE365/checkpoints/{run_id}'
os.environ['DRIVE_CKPT'] = drive_ckpt_path

# Tạo thư mục con checkpoints và thư mục run_id bằng os.makedirs
os.makedirs(drive_ckpt_path, exist_ok=True)
print(f'Mọi checkpoint trong phiên này sẽ được lưu chung vào: {drive_ckpt_path}')

### BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
> ✏️ **Chỗ cần sửa (Tuỳ chọn):** Bạn có thể thay đổi tham số `--epochs 1` thành số vòng lặp bạn muốn (mặc định là 5).


In [ ]:
# BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
!python main.py --mode train_text --epochs 1

# Lưu Checkpoint Text ngay lập tức
!cp ./checkpoints/best_model_train_text.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Text vào $DRIVE_CKPT"

### BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)
> ✏️ **Chỗ cần sửa (Tuỳ chọn):** Tương tự, thay đổi `--epochs` hoặc `--batch_size` nếu cần.


In [ ]:
# BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)
!python main.py --mode train_image --epochs 1

# Lưu Checkpoint Image ngay lập tức
!cp ./checkpoints/best_model_train_image.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Image vào $DRIVE_CKPT"

### BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
Mô hình này sẽ tự động tải các checkpoint tốt nhất của mô hình Text và Image vừa train ở trên để tiếp tục huấn luyện phần kết hợp.
> ✏️ **Chỗ cần sửa (Tuỳ chọn):** Bạn có thể chỉnh sửa tham số `--alpha` (Trọng số Loss) ở lệnh này.


In [ ]:
# BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
!python main.py --mode train_fusion --epochs 1

# Lưu Checkpoint Fusion ngay lập tức
!cp ./checkpoints/best_model_train_fusion.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Fusion vào $DRIVE_CKPT"

### BƯỚC 7: Đánh giá (Test) trên mô hình tốt nhất
Bước cuối cùng là chạy đánh giá mô hình Fusion trên tập dữ liệu Test chưa từng được thấy để tính ra các chỉ số sai số MAE, MSE và RMSE.


In [ ]:
# BƯỚC 7: Đánh giá mô hình (Tính các metric MAE, MSE, RMSE)
!python test.py --mode train_fusion